<a href="https://colab.research.google.com/github/madeeha-noor/Student-Success-Early-Warning/blob/main/notebooks/02_data_quality_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install -q ucimlrepo

In [10]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ds = fetch_ucirepo(id = 697)
df = pd.concat([ds.data.features, ds.data.targets], axis = 1)
df.columns = df.columns.str.strip()
print(df.shape)

(4424, 37)


In [11]:
print("Missing values:", df.isna(). sum(). sum())
print("Duplicate rows:", df.duplicated(). sum())
df.describe().T

Missing values: 0
Duplicate rows: 0


,count,mean,std,min,25%,50%,75%,max
Marital Status,4424.0,1.178571,0.605747,1.00,1.00,1.000000,1.000000,6.000000
Application mode,4424.0,18.669078,17.484682,1.00,1.00,17.000000,39.000000,57.000000
Application order,4424.0,1.727848,1.313793,0.00,1.00,1.000000,2.000000,9.000000
Course,4424.0,8856.642631,2063.566416,33.00,9085.00,9238.000000,9556.000000,9991.000000
Daytime/evening attendance,4424.0,0.890823,0.311897,0.00,1.00,1.000000,1.000000,1.000000
Previous qualification,4424.0,4.577758,10.216592,1.00,1.00,1.000000,1.000000,43.000000
Previous qualification (grade),4424.0,132.613314,13.188332,95.00,125.00,133.100000,140.000000,190.000000
Nacionality,4424.0,1.873192,6.914514,1.00,1.00,1.000000,1.000000,109.000000
Mother's qualification,4424.0,19.561935,15.603186,1.00,2.00,19.000000,37.000000,44.000000
Father's qualification,4424.0,22.275316,15.343108,1.00,3.00,19.000000,37.000000,44.000000


## Summary Statistics: what i noticed
- All 36 feature columns have 4,424 vaues, so there are no missing values
- olumns such as Course, Martial Status, Application mode, Nacionality and the parents' qualiffication / occupation are category codes, not real quantites. We will decode them and treat them as categories
- Age at enrollment is skewed: median 20, maximum 70
- Ciurricular unit counts have maximums far above the 75th percentile (e.g: 1st sem evaluations: 75% = 10, max = 45), so they are possible outliers
- Grades column have a minimum of 0. We still need to check whether these are students with no units, not real zero scores

In [12]:
for sem in ["1st", "2nd"]:
  enrolled = df[f"Curricular units {sem} sem (enrolled)"]
  grade = df[f"Curricular units {sem} sem (grade)"]
  print(f"{sem} sem -> 0 units enrolled: {(enrolled == 0).sum()}, grade is 0:{(grade == 0).sum()}, both: {((enrolled == 0) & (grade == 0)).sum()}")

1st sem -> 0 units enrolled: 180, grade is 0:718, both: 180
2nd sem -> 0 units enrolled: 180, grade is 0:870, both: 180


In [13]:
df.nunique().sort_values()

,0
Daytime/evening attendance,2
Displaced,2
Debtor,2
Educational special needs,2
International,2
Scholarship holder,2
Gender,2
Tuition fees up to date,2
Target,3
Marital Status,6


In [14]:
zero_both = df[(df["Curricular units 1st sem (enrolled)"] == 0) & (df["Curricular units 2nd sem (enrolled)"] == 0)]
print("Students with 0 units in BOTH semesters:", len(zero_both))
print(zero_both["Target"].value_counts())

Students with 0 units in BOTH semesters: 180
Target
Dropout     77
Graduate    75
Enrolled    28
Name: count, dtype: int64


In [15]:
for sem in ["1st", "2nd"]:
  enr = df[f"Curricular units {sem} sem (enrolled)"]
  gr = df[f"Curricular units {sem} sem (grade)"]
  sub = df[(enr > 0) & (gr == 0)]
  print(f"{sem} sem: enrolled but grade 0 = {len(sub)}; "
        f"With 0 evaluations = {(sub[f"Curricular units {sem} sem (evaluations)"] == 0).sum()}; "
        f"With 0 approved = {(sub[f"Curricular units {sem} sem (approved)"] == 0).sum()}")

1st sem: enrolled but grade 0 = 538; With 0 evaluations = 169; With 0 approved = 538
2nd sem: enrolled but grade 0 = 690; With 0 evaluations = 221; With 0 approved = 690


In [17]:
c1 = zero_both["Curricular units 1st sem (credited)"]
c2 = zero_both["Curricular units 2nd sem (credited)"]
print("Zero-unit students with some credited units:", ((c1 > 0) | (c2 > 0)).sum())

Zero-unit students with some credited units: 0


In [18]:
for sem in ["1st", "2nd"]:
  enr = df[f"Curricular units {sem} sem (enrolled)"]
  app = df[f"Curricular units {sem} sem (approved)"]
  ev = df[f"Curricular units {sem} sem (evaluations)"]
  print(f"{sem} sem: approved > enrolled: {(app > enr).sum()}, "
        f"approved > evaluations: {(app > ev).sum()}, "
        f"evaluations > 0 but enrolled == 0: {((ev > 0) & (enr == 0)).sum()}")

1st sem: approved > enrolled: 0, approved > evaluations: 0, evaluations > 0 but enrolled == 0: 0
2nd sem: approved > enrolled: 0, approved > evaluations: 0, evaluations > 0 but enrolled == 0: 0
